In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

In [6]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

# ==========================================
# 1. Projection of a 3D point
# ==========================================
print("--- Part 1: Projection of a 3D point ---\n")

# Camera intrinsics
fx, fy = 480, 480
cx, cy = 320, 270

K = np.array([
    [fx,  0, cx],
    [ 0, fy, cy],
    [ 0,  0,  1]
])

# Pose parameters given in global frame
R_G_C = np.array([
    [ 0.5363, -0.8440, 0],
    [ 0.8440,  0.5363, 0],
    [ 0,       0,      1]
])

t_C_to_G_global = np.array([-451.2459, 257.0322, 400]).reshape(3, 1)
X_G = np.array([350, -250, -35]).reshape(3, 1)

# The point in the camera frame is X^C = R_G^C * (X^G - t_C_to_G_global)
# P = K * [R | t] where [R|t] is the mapping from Global to Camera.
# R = R_G^C
# t = -R_G^C * t_C_to_G_global
t_cam = -R_G_C @ t_C_to_G_global

# Extrinsic matrix
Rt = np.hstack((R_G_C, t_cam))

# (a) Camera projection matrix P
P = K @ Rt
print("(a) Projection Matrix P:")
print(np.round(P, 4), "\n")

# (b) Projecting the 3D point X_G
# Homogeneous 3D point
X_G_homog = np.vstack((X_G, [[1]]))
x_img_homog = P @ X_G_homog

# Normalize by Z to get pixel coordinates
u_proj = x_img_homog[0, 0] / x_img_homog[2, 0]
v_proj = x_img_homog[1, 0] / x_img_homog[2, 0]
print(f"(b) Projected Image Coordinates: (u, v) = ({u_proj:.4f}, {v_proj:.4f})\n")

# (c) Re-projection error
obs_u, obs_v = 241.5, 169.0
err_u = u_proj - obs_u
err_v = v_proj - obs_v
reproj_error = np.sqrt(err_u**2 + err_v**2)

print(f"(c) Re-projection Error: {reproj_error:.4f} pixels\n")


# ==========================================
# 2. Camera Calibration (Framework)
# ==========================================
# Note: Since this requires physical images, plug your image paths below.
print("--- Part 2: Calibrate your own camera (Framework) ---\n")

def calibrate_camera(image_paths, checkerboard_size=(8, 6)):
    """
    Standard OpenCV implementation for Camera Calibration.
    Requires an array of file paths to images of a checkerboard pattern.
    """
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
    objp = np.zeros((checkerboard_size[0]*checkerboard_size[1], 3), np.float32)
    objp[:, :2] = np.mgrid[0:checkerboard_size[0], 0:checkerboard_size[1]].T.reshape(-1, 2)

    objpoints = [] # 3d point in real world space
    imgpoints = [] # 2d points in image plane

    for fname in image_paths:
        img = cv2.imread(fname)
        if img is None: continue
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        ret, corners = cv2.findChessboardCorners(gray, checkerboard_size, None)
        if ret:
            objpoints.append(objp)
            corners2 = cv2.cornerSubPix(gray, corners, (11,11), (-1,-1), criteria)
            imgpoints.append(corners2)

    # Calibrate
    if objpoints:
        ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, gray.shape[::-1], None, None)
        print("Camera Calibration Matrix (K):\n", mtx)
        print(f"\nPrincipal Point: ({mtx[0,2]:.2f}, {mtx[1,2]:.2f})")
        print(f"Focal Length: fx={mtx[0,0]:.2f}, fy={mtx[1,1]:.2f}\n")
        return mtx, dist
    else:
        print("No checkerboard corners found. Update image paths.")
        return None, None

# Example usage (Uncomment when running with real images)
# image_paths = ['calib1.jpg', 'calib2.jpg', 'calib3.jpg']
# K_estimated, _ = calibrate_camera(image_paths)


# ==========================================
# 3. Image Feature Extraction & Matching
# ==========================================
# OpenCV provides native SIFT alternatives to vlfeat.
print("--- Part 3: SIFT Extraction and Matching (Framework) ---\n")

def extract_and_match_sift(img1_path, img2_path):
    img1 = cv2.imread(img1_path, cv2.IMREAD_GRAYSCALE)
    img2 = cv2.imread(img2_path, cv2.IMREAD_GRAYSCALE)

    if img1 is None or img2 is None:
        print("Please provide valid paths to your selfies.")
        return

    # (b) Extract SIFT features
    sift = cv2.SIFT_create()
    kp1, des1 = sift.detectAndCompute(img1, None)
    kp2, des2 = sift.detectAndCompute(img2, None)

    print(f"Extracted {len(kp1)} features from image 1.")
    if len(kp1) > 0:
        rep_kp = kp1[0]
        print(f"Representative feature (Img 1) - Scale/Size: {rep_kp.size:.2f}, Orientation: {rep_kp.angle:.2f} degrees")

    # Plot features
    img1_kp = cv2.drawKeypoints(img1, kp1, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
    img2_kp = cv2.drawKeypoints(img2, kp2, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
    ax1.imshow(img1_kp); ax1.set_title("Image 1 SIFT Features")
    ax2.imshow(img2_kp); ax2.set_title("Image 2 SIFT Features")
    plt.show()

    # (c) Calculate putative matches using BFMatcher
    bf = cv2.BFMatcher()
    matches = bf.knnMatch(des1, des2, k=2)

    # Apply ratio test (Lowe's ratio) to distinguish inliers
    good_matches = []
    outliers = []
    for m, n in matches:
        if m.distance < 0.75 * n.distance:
            good_matches.append([m])
        else:
            outliers.append([m])

    # Draw representative matches (Show top 20 good ones)
    img_matches = cv2.drawMatchesKnn(img1, kp1, img2, kp2, good_matches[:20], None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

    plt.figure(figsize=(12, 6))
    plt.imshow(img_matches)
    plt.title("Calculated Putative Matches (Inliers shown)")
    plt.show()

    print(f"Total Matches: {len(matches)}")
    print(f"Inliers (passed ratio test): {len(good_matches)}")
    print(f"Outliers (failed ratio test): {len(outliers)}")

# Example usage (Uncomment when running with real images)
# extract_and_match_sift('selfie_yaw_0.jpg', 'selfie_yaw_30.jpg')

--- Part 1: Projection of a 3D point ---

(a) Projection Matrix P:
[[ 2.57424000e+02 -4.05120000e+02  3.20000000e+02  9.22904094e+04]
 [ 4.05120000e+02  2.57424000e+02  2.70000000e+02  8.64248200e+03]
 [ 0.00000000e+00  0.00000000e+00  1.00000000e+00 -4.00000000e+02]] 

(b) Projected Image Coordinates: (u, v) = (-626.3651, -176.1574)

(c) Re-projection Error: 933.9826 pixels

--- Part 2: Calibrate your own camera (Framework) ---

--- Part 3: SIFT Extraction and Matching (Framework) ---



In [7]:
# %% [markdown]
# # Question 1: Projection of a 3D point
# %%
import numpy as np

# Camera intrinsics
f = 480
cx, cy = 320, 270
K = np.array([
    [f, 0, cx],
    [0, f, cy],
    [0, 0, 1]
])

# Rotation and translation (Global to Camera mapping given in the problem)
R_G_C = np.array([
    [0.5363, -0.8440, 0],
    [0.8440, 0.5363, 0],
    [0, 0, 1]
])

t_C_G = np.array([[-451.2459], [257.0322], [400]])
X_G = np.array([[350], [-250], [-35]])

# (a) Expression for Camera Projection Matrix P
# Assuming R_G_C transforms points from Global to Camera frame: X_C = R_G_C @ (X_G - t_C_G)
# P = K @ [R_G_C | -R_G_C @ t_C_G]
t_G_C = -R_G_C @ t_C_G
P = K @ np.hstack((R_G_C, t_G_C))

print("(a) Camera Projection Matrix P:")
print(np.round(P, 4))

# (b) Projecting the 3D point
# X_C = R_G_C * (X_G - t_C_G)
X_C = R_G_C @ (X_G - t_C_G)

# Project to image plane
x_img_homog = K @ X_C
u = x_img_homog[0, 0] / x_img_homog[2, 0]
v = x_img_homog[1, 0] / x_img_homog[2, 0]

print(f"\n(b) Projected Image Coordinates (u, v): ({u:.4f}, {v:.4f})")

# (c) Re-projection Error
obs_u, obs_v = 241.5, 169
error = np.sqrt((u - obs_u)**2 + (v - obs_v)**2)
print(f"\n(c) Re-projection Error: {error:.4f} pixels")

(a) Camera Projection Matrix P:
[[ 2.57424000e+02 -4.05120000e+02  3.20000000e+02  9.22904094e+04]
 [ 4.05120000e+02  2.57424000e+02  2.70000000e+02  8.64248200e+03]
 [ 0.00000000e+00  0.00000000e+00  1.00000000e+00 -4.00000000e+02]]

(b) Projected Image Coordinates (u, v): (-626.3651, -176.1574)

(c) Re-projection Error: 933.9826 pixels


In [8]:
# %% [markdown]
# # Question 2: Camera Calibration
# Take ~10-15 pictures of a standard checkerboard pattern from various angles using your phone/webcam.
# Save them in a folder named 'calibration_images'.
# %%
import cv2
import glob

# Checkerboard dimensions (inner corners per row and column)
CHECKERBOARD = (6, 9)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

# 3D points in real world space
objp = np.zeros((CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2)

objpoints = [] # 3d points in real world space
imgpoints = [] # 2d points in image plane.

images = glob.glob('calibration_images/*.jpg')

for fname in images:
    img = cv2.imread(fname)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Find the chess board corners
    ret, corners = cv2.findChessboardCorners(gray, CHECKERBOARD, None)
    if ret:
        objpoints.append(objp)
        corners2 = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
        imgpoints.append(corners2)

        # (c) Draw and display the corners (Snapshot output)
        cv2.drawChessboardCorners(img, CHECKERBOARD, corners2, ret)
        cv2.imshow('Calibration Snapshot', img)
        cv2.waitKey(500)

cv2.destroyAllWindows()

# (d) Camera calibration matrix K
ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, gray.shape[::-1], None, None)

print("(d) Estimated Camera Calibration Matrix K:")
print(np.round(mtx, 2))
print(f"Focal Length (fx, fy): ({mtx[0,0]:.2f}, {mtx[1,1]:.2f})")
print(f"Principal Point (cx, cy): ({mtx[0,2]:.2f}, {mtx[1,2]:.2f})")

NameError: name 'gray' is not defined

In [10]:
# %% [markdown]
# # Question 2: Camera Calibration
# Take ~10-15 pictures of a standard checkerboard pattern from various angles using your phone/webcam.
# Save them in a folder named 'calibration_images'.
# %%
import cv2
import glob

# Checkerboard dimensions (inner corners per row and column)
CHECKERBOARD = (6, 9)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

# 3D points in real world space
objp = np.zeros((CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2)

objpoints = [] # 3d points in real world space
imgpoints = [] # 2d points in image plane.

images = glob.glob('calibration_images/*.jpg')

for fname in images:
    img = cv2.imread(fname)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Find the chess board corners
    ret, corners = cv2.findChessboardCorners(gray, CHECKERBOARD, None)
    if ret:
        objpoints.append(objp)
        corners2 = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
        imgpoints.append(corners2)

        # (c) Draw and display the corners (Snapshot output)
        cv2.drawChessboardCorners(img, CHECKERBOARD, corners2, ret)
        cv2.imshow('Calibration Snapshot', img)
        cv2.waitKey(500)

cv2.destroyAllWindows()

# (d) Camera calibration matrix K
ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, gray.shape[::-1], None, None)

print("(d) Estimated Camera Calibration Matrix K:")
print(np.round(mtx, 2))
print(f"Focal Length (fx, fy): ({mtx[0,0]:.2f}, {mtx[1,1]:.2f})")
print(f"Principal Point (cx, cy): ({mtx[0,2]:.2f}, {mtx[1,2]:.2f})")

NameError: name 'gray' is not defined